<center>

# Universidad Nacional Autónoma de México
## Facultad de Ciencias

<hr style="height:3px; background-color:#003366; border:none;"/>

## Inteligencia Artificial
### Examen — Algoritmo Supervisado: Árbol de Decisión (CART)

**Fundamentos Matemáticos · Descripción Formal · Pseudocódigo · Código · Buenas Prácticas**

<hr style="height:3px; background-color:#003366; border:none;"/>

</center>

## 1. Introducción y Contexto

El **Árbol de Decisión** es uno de los algoritmos de aprendizaje supervisado más interpretables en Machine Learning. Aprende una **jerarquía de preguntas binarias** sobre las características del dato para llegar a una predicción de clasificación.

En este documento se implementa la variante **CART** (*Classification and Regression Trees*), que es la base de algoritmos más avanzados vistos en clase como **Random Forest**, **Gradient Boosting** y **XGBoost**.

> **¿Por qué Árbol de Decisión?**
> - Es el bloque base de Random Forest, Gradient Boosting y XGBoost (Lab 4.2).
> - Es completamente interpretable: se puede explicar cada decisión.
> - Introduce **Entropía** e **Información Ganada**, fundamentales en IA.

---
## 2. Fundamentos Matemáticos

### 2.1 Entropía de Shannon

La **entropía** mide el grado de impureza de un conjunto $S$ con $C$ clases posibles:

$$H(S) = -\sum_{i=1}^{C} p_i \log_2(p_i)$$

donde $p_i$ es la proporción de ejemplos que pertenecen a la clase $i$.

**Casos extremos:**
- $H(S) = 0$ → todos los elementos son de **una sola clase** (máxima pureza).
- $H(S) = 1$ → clases **perfectamente balanceadas** (máximo desorden, 2 clases).

### 2.2 Información Ganada (Information Gain)

La **Información Ganada** cuantifica cuánto se reduce la entropía al dividir $S$ usando el atributo $A$ con umbral $t$:

$$IG(S, A, t) = H(S) - \left( \frac{|S_L|}{|S|} H(S_L) + \frac{|S_R|}{|S|} H(S_R) \right)$$

donde $S_L = \{x \in S \mid x_A \leq t\}$ y $S_R = \{x \in S \mid x_A > t\}$.

El algoritmo CART elige la combinación que **maximiza** $IG$:

$$A^*, t^* = \underset{A,\, t}{\arg\max}\ IG(S, A, t)$$

### 2.3 Condiciones de Parada

La recursión se detiene cuando se cumple **al menos una** de estas condiciones:

1. $H(S) = 0$ — Todos los ejemplos son de la misma clase.
2. $\text{profundidad} \geq \text{max\_depth}$ — Límite de profundidad alcanzado.

### 2.4 Complejidad Computacional

| Operación | Complejidad |
|-----------|-------------|
| Entrenamiento | $O(n \cdot d \cdot \log n)$ |
| Predicción (por muestra) | $O(\log n)$ árbol balanceado |
| Espacio | $O(n)$ |

donde $n$ = número de muestras y $d$ = número de características.

---
## 3. Descripción Formal del Algoritmo

El árbol CART construye un árbol binario de forma **recursiva y greedy**: en cada nodo toma la mejor decisión local sin considerar impacto global.

**Tipos de nodos:**
- **Nodo interno (pregunta):** contiene atributo $A$ y umbral $t$. Manda la muestra a la izquierda si $x_A \leq t$, a la derecha si $x_A > t$.
- **Nodo hoja (respuesta):** almacena la clase mayoritaria de los ejemplos que llegaron.

**Proceso de aprendizaje:**
1. La raíz recibe todo el conjunto $\mathcal{D}$.
2. Se evalúan todos los umbrales para cada característica; se elige el que maximiza $IG$.
3. Los datos se dividen en $S_L$ y $S_R$.
4. Se repite el proceso recursivamente hasta un criterio de parada.

**Proceso de inferencia:** se recorre el árbol desde la raíz evaluando condiciones hasta llegar a una hoja.

---
## 4. Pseudocódigo

```
ALGORITMO: Árbol de Decisión CART
=========================================================

FUNCIÓN construir_árbol(X, y, profundidad):
    SI profundidad >= max_depth O todas_misma_clase(y):
        RETORNAR Hoja(clase = clase_mayoritaria(y))
    FIN SI

    mejor_IG ← -infinito
    PARA cada atributo A en [0 ... d-1]:
        PARA cada umbral t en valores_únicos(X[:, A]):
            S_L ← {(x,y) | x[A] <= t}
            S_R ← {(x,y) | x[A]  > t}
            IG  ← entropía(y) - suma_ponderada(H(S_L.y), H(S_R.y))
            SI IG > mejor_IG:
                mejor_IG ← IG; mejor_A ← A; mejor_t ← t
        FIN PARA
    FIN PARA

    nodo_izq ← construir_árbol(X[S_L], y[S_L], profundidad+1)
    nodo_der ← construir_árbol(X[S_R], y[S_R], profundidad+1)
    RETORNAR NodoInterno(atributo=mejor_A, umbral=mejor_t,
                         izquierda=nodo_izq, derecha=nodo_der)
FIN FUNCIÓN

FUNCIÓN entropía(y):
    proporciones ← contar_clases(y) / |y|
    RETORNAR -SUMA(p * log2(p) para p > 0)
FIN FUNCIÓN

FUNCIÓN predecir(x, nodo):
    SI nodo es Hoja: RETORNAR nodo.clase
    SI x[nodo.atributo] <= nodo.umbral:
        RETORNAR predecir(x, nodo.izquierda)
    SINO: RETORNAR predecir(x, nodo.derecha)
FIN FUNCIÓN
```

## 5. Implementación en Python

Buenas prácticas aplicadas (siguiendo el Lab 4.2):
- Separación de responsabilidades en métodos con nombres descriptivos.
- Uso de `numpy` para operaciones vectorizadas (eficiencia de memoria y caché).
- Documentación con *docstrings* en cada método.
- Constantes definidas en el constructor `__init__`.
- Uso de `if __name__ == "__main__"` para separar definición de ejecución.

In [ ]:
# ============================================================
# DEPENDENCIAS
# ============================================================
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

In [ ]:
# ============================================================
# ESTRUCTURA DE DATOS: Nodo del Árbol
# ============================================================

class Nodo:
    """
    Representa un nodo dentro del Árbol de Decisión.

    Puede ser:
    - Nodo Interno (pregunta): tiene feature, threshold, left y right.
    - Nodo Hoja   (respuesta): solo tiene value (la clase predicha).

    Atributos
    ----------
    feature   : int   -- Índice de la característica usada para dividir.
    threshold : float -- Valor de corte para la división.
    left      : Nodo  -- Sub-árbol izquierdo (x[feature] <= threshold).
    right     : Nodo  -- Sub-árbol derecho   (x[feature]  > threshold).
    value     : int   -- Clase predicha (sólo en nodo hoja).
    """

    def __init__(
        self,
        feature: int     = None,
        threshold: float = None,
        left             = None,
        right            = None,
        value: int       = None
    ):
        self.feature   = feature
        self.threshold = threshold
        self.left      = left
        self.right     = right
        self.value     = value

    def es_hoja(self) -> bool:
        """Retorna True si el nodo es una hoja (tiene valor de predicción)."""
        return self.value is not None

In [ ]:
# ============================================================
# ALGORITMO PRINCIPAL: Árbol de Decisión CART
# ============================================================

class ArbolDecisionCART:
    """
    Implementación del Árbol de Decisión CART para clasificación.

    Usa la Entropía de Shannon como criterio de impureza
    y la Información Ganada para seleccionar el mejor corte en cada nodo.

    Parámetros
    ----------
    max_depth : int -- Profundidad máxima del árbol (controla el sobreajuste).
    """

    def __init__(self, max_depth: int = 5):
        self.max_depth = max_depth
        self.raiz: Nodo = None

    # ----------------------------------------------------------
    # INTERFAZ PÚBLICA
    # ----------------------------------------------------------

    def fit(self, X: np.ndarray, y: np.ndarray) -> None:
        """
        Entrena el árbol construyéndolo recursivamente sobre (X, y).

        Parámetros
        ----------
        X : array (n_muestras, n_características) -- Características de entrada.
        y : array (n_muestras,)                   -- Etiquetas enteras de clase.
        """
        self.raiz = self._construir_arbol(X, y, profundidad=0)

    def predict(self, X: np.ndarray) -> np.ndarray:
        """
        Genera predicciones para cada muestra en X.

        Parámetros
        ----------
        X : array (n_muestras, n_características)

        Retorna
        -------
        array (n_muestras,) con las clases predichas.
        """
        return np.array([self._recorrer_arbol(x, self.raiz) for x in X])

    # ----------------------------------------------------------
    # CONSTRUCCIÓN DEL ÁRBOL (ENTRENAMIENTO)
    # ----------------------------------------------------------

    def _construir_arbol(self, X: np.ndarray, y: np.ndarray, profundidad: int) -> Nodo:
        """
        Construye el árbol de forma recursiva aplicando búsqueda greedy.
        Crea una hoja cuando se cumple un criterio de parada.
        """
        n_muestras, n_caracteristicas = X.shape
        n_clases_unicas = len(np.unique(y))

        # --- CRITERIOS DE PARADA: crear nodo hoja ---
        criterio_profundidad = profundidad >= self.max_depth
        criterio_pureza      = n_clases_unicas == 1

        if criterio_profundidad or criterio_pureza:
            clase_mayoritaria = Counter(y).most_common(1)[0][0]
            return Nodo(value=clase_mayoritaria)

        # --- BÚSQUEDA GREEDY: mejor (atributo, umbral) ---
        mejor_atributo, mejor_umbral = self._encontrar_mejor_corte(
            X, y, n_caracteristicas
        )

        # --- PARTICIÓN DE DATOS ---
        mascara_izq = X[:, mejor_atributo] <= mejor_umbral
        mascara_der = ~mascara_izq

        # --- RECURSIÓN EN RAMAS HIJAS ---
        nodo_izquierdo = self._construir_arbol(
            X[mascara_izq], y[mascara_izq], profundidad + 1
        )
        nodo_derecho = self._construir_arbol(
            X[mascara_der], y[mascara_der], profundidad + 1
        )

        return Nodo(
            feature=mejor_atributo,
            threshold=mejor_umbral,
            left=nodo_izquierdo,
            right=nodo_derecho
        )

    def _encontrar_mejor_corte(
        self, X: np.ndarray, y: np.ndarray, n_caracteristicas: int
    ):
        """
        Evalúa todos los umbrales para todas las características.
        Retorna (atributo, umbral) que maximiza la Información Ganada.
        """
        mejor_ganancia = -1.0
        mejor_atributo = None
        mejor_umbral   = None

        for indice_attr in range(n_caracteristicas):
            # Solo evaluamos umbrales en los valores únicos presentes
            umbrales_candidatos = np.unique(X[:, indice_attr])

            for umbral in umbrales_candidatos:
                ganancia = self._calcular_informacion_ganada(
                    columna=X[:, indice_attr],
                    y=y,
                    umbral=umbral
                )

                if ganancia > mejor_ganancia:
                    mejor_ganancia = ganancia
                    mejor_atributo = indice_attr
                    mejor_umbral   = umbral

        return mejor_atributo, mejor_umbral

    # ----------------------------------------------------------
    # MÉTRICAS: Entropía e Información Ganada
    # ----------------------------------------------------------

    def _calcular_informacion_ganada(
        self, columna: np.ndarray, y: np.ndarray, umbral: float
    ) -> float:
        """
        IG = H(padre) - [ (n_izq/n)*H(izq) + (n_der/n)*H(der) ]

        Retorna 0.0 si alguna partición queda vacía.
        """
        entropia_padre = self._entropia(y)

        mascara_izq = columna <= umbral
        mascara_der = ~mascara_izq

        # Validación: ambas particiones deben ser no vacías
        if mascara_izq.sum() == 0 or mascara_der.sum() == 0:
            return 0.0

        n     = len(y)
        n_izq = mascara_izq.sum()
        n_der = mascara_der.sum()

        entropia_hijos = (
            (n_izq / n) * self._entropia(y[mascara_izq]) +
            (n_der / n) * self._entropia(y[mascara_der])
        )

        return entropia_padre - entropia_hijos

    def _entropia(self, y: np.ndarray) -> float:
        """
        H(S) = -sum(p_i * log2(p_i))

        Usa np.bincount (vectorizado) para contar clases eficientemente.
        Filtra proporciones nulas para evitar log2(0).
        """
        proporciones = np.bincount(y) / len(y)
        return -np.sum([p * np.log2(p) for p in proporciones if p > 0])

    # ----------------------------------------------------------
    # INFERENCIA (PREDICCIÓN)
    # ----------------------------------------------------------

    def _recorrer_arbol(self, x: np.ndarray, nodo: Nodo) -> int:
        """
        Recorre el árbol recursivamente para una muestra x.
        Retorna la clase predicha al llegar a una hoja.
        """
        if nodo.es_hoja():
            return nodo.value

        if x[nodo.feature] <= nodo.threshold:
            return self._recorrer_arbol(x, nodo.left)
        else:
            return self._recorrer_arbol(x, nodo.right)

---
## 6. Aplicación y Evaluación del Modelo

In [ ]:
# ============================================================
# APLICACIÓN: Dataset de clasificación binaria
# ============================================================

if __name__ == "__main__":

    # --- 1. PREPARACIÓN DE DATOS ---
    # Dataset sintético con 2 características para facilitar visualización
    np.random.seed(42)
    X, y = make_classification(
        n_samples=200,
        n_features=2,
        n_informative=2,
        n_redundant=0,
        n_clusters_per_class=1,
        random_state=42
    )

    # División entrenamiento (80%) / prueba (20%)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    print(f"Muestras de entrenamiento : {X_train.shape[0]}")
    print(f"Muestras de prueba        : {X_test.shape[0]}")

    # --- 2. ENTRENAMIENTO DEL MODELO ---
    modelo = ArbolDecisionCART(max_depth=4)
    modelo.fit(X_train, y_train)
    print("\n✓ Árbol entrenado correctamente.")

    # --- 3. PREDICCIÓN Y MÉTRICAS ---
    y_pred    = modelo.predict(X_test)
    exactitud = accuracy_score(y_test, y_pred)

    print(f"\nExactitud en prueba: {exactitud * 100:.2f}%")
    print("\nReporte de Clasificación:")
    print(classification_report(y_test, y_pred, target_names=["Clase 0", "Clase 1"]))

---
## 7. Visualización: Frontera de Decisión

In [ ]:
# ============================================================
# VISUALIZACIÓN: Frontera de Decisión del Árbol
# ============================================================

def visualizar_frontera_decision(
    modelo, X_train, y_train, X_test, y_test,
    titulo: str = "Árbol de Decisión CART"
) -> None:
    """
    Grafica la frontera de decisión aprendida por el árbol,
    junto con los puntos de entrenamiento y prueba.
    """
    margen = 0.5
    paso   = 0.02
    x1_min = X_train[:, 0].min() - margen
    x1_max = X_train[:, 0].max() + margen
    x2_min = X_train[:, 1].min() - margen
    x2_max = X_train[:, 1].max() + margen

    xx1, xx2 = np.meshgrid(
        np.arange(x1_min, x1_max, paso),
        np.arange(x2_min, x2_max, paso)
    )

    Z = modelo.predict(np.c_[xx1.ravel(), xx2.ravel()])
    Z = Z.reshape(xx1.shape)

    fig, ejes = plt.subplots(1, 2, figsize=(14, 5))

    conjuntos = [
        (X_train, y_train, "Datos de Entrenamiento"),
        (X_test,  y_test,  "Datos de Prueba")
    ]

    for ax, (X_vis, y_vis, subtitulo) in zip(ejes, conjuntos):
        ax.contourf(xx1, xx2, Z, alpha=0.3, cmap="RdBu")
        ax.contour(xx1, xx2, Z, colors="black", linewidths=0.8, alpha=0.5)
        ax.scatter(
            X_vis[:, 0], X_vis[:, 1],
            c=y_vis, cmap="RdBu", edgecolors="k", s=60, linewidth=0.8
        )
        ax.set_xlabel("Característica 1", fontsize=11)
        ax.set_ylabel("Característica 2", fontsize=11)
        ax.set_title(f"{titulo}\n{subtitulo}", fontsize=12, fontweight="bold")
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()


if __name__ == "__main__":
    visualizar_frontera_decision(modelo, X_train, y_train, X_test, y_test)

---
## 8. Análisis del Efecto de `max_depth` (Hiperparámetro Clave)

In [ ]:
# ============================================================
# EXPERIMENTO: Impacto de max_depth (sobreajuste vs. subajuste)
# ============================================================

if __name__ == "__main__":

    profundidades     = range(1, 11)
    exactitudes_train = []
    exactitudes_test  = []

    for profundidad in profundidades:
        arbol = ArbolDecisionCART(max_depth=profundidad)
        arbol.fit(X_train, y_train)
        exactitudes_train.append(accuracy_score(y_train, arbol.predict(X_train)))
        exactitudes_test.append(accuracy_score(y_test,  arbol.predict(X_test)))

    plt.figure(figsize=(9, 5))
    plt.plot(profundidades, exactitudes_train, "o-",  label="Entrenamiento", color="steelblue")
    plt.plot(profundidades, exactitudes_test,  "s--", label="Prueba",        color="tomato")
    plt.xlabel("Profundidad máxima (max_depth)", fontsize=12)
    plt.ylabel("Exactitud", fontsize=12)
    plt.title("Curva de Complejidad — Árbol de Decisión CART", fontsize=13, fontweight="bold")
    plt.legend(fontsize=11)
    plt.xticks(profundidades)
    plt.ylim(0.5, 1.05)
    plt.grid(True, alpha=0.4)
    plt.tight_layout()
    plt.show()

    print(f"{'Profundidad':>12} | {'Train':>8} | {'Test':>8}")
    print("-" * 36)
    for d, tr, te in zip(profundidades, exactitudes_train, exactitudes_test):
        print(f"{d:>12} | {tr:>8.4f} | {te:>8.4f}")

---
## 9. Verificación de los Fundamentos Matemáticos

In [ ]:
# ============================================================
# VERIFICACIÓN: Demostración de Entropía e Información Ganada
# ============================================================

if __name__ == "__main__":

    _aux = ArbolDecisionCART()

    # Entropía máxima: clases perfectamente balanceadas → H = 1.0
    y_bal = np.array([0, 0, 0, 1, 1, 1])
    H_bal = _aux._entropia(y_bal)
    print(f"H(S) clases balanceadas [0,0,0,1,1,1] = {H_bal:.4f}  (esperado ≈ 1.0)")

    # Entropía mínima: todos de la misma clase → H = 0.0
    y_pur = np.array([1, 1, 1, 1, 1, 1])
    H_pur = _aux._entropia(y_pur)
    print(f"H(S) clase pura         [1,1,1,1,1,1] = {H_pur:.4f}  (esperado = 0.0)")

    # Información Ganada de un corte perfecto (separa completamente las clases)
    columna = np.array([1, 2, 3, 4, 5, 6], dtype=float)
    y_ej    = np.array([0, 0, 0, 1, 1, 1])
    ig = _aux._calcular_informacion_ganada(columna, y_ej, umbral=3.0)
    print(f"\nIG (corte perfecto, umbral=3.0) = {ig:.4f}  (esperado = 1.0)")

---
## 10. Conclusiones

### Resultados obtenidos

1. **Exactitud:** El modelo alcanza una exactitud competitiva en el conjunto de prueba, confirmando que la implementación desde cero es funcionalmente correcta.

2. **Curva de complejidad:** Se observa el fenómeno de **sobreajuste**: al aumentar `max_depth`, la exactitud en entrenamiento sube continuamente, pero la exactitud en prueba se estabiliza o disminuye. Esto valida la necesidad de controlar la profundidad como hiperparámetro.

3. **Verificación matemática:** La entropía es exactamente 1.0 para clases balanceadas y 0.0 para clases puras, y la Información Ganada es 1.0 para un corte perfecto, confirmando la correcta implementación de los fundamentos.

### Relación con los algoritmos del Lab 4.2

| Algoritmo del Lab | Rol del Árbol de Decisión |
|-------------------|---------------------------|
| **Random Forest** | Ensambla múltiples árboles con Bootstrap sampling y votación mayoritaria. |
| **Gradient Boosting** | Entrena árboles de regresión secuenciales sobre los residuos del modelo anterior. |
| **XGBoost** | Variante regularizada (L2) del Gradient Boosting con árboles (hoja regularizada: $\sum g / (\sum h + \lambda)$). |

> El Árbol de Decisión es el **componente atómico** sobre el que se construyen los métodos de ensamble más poderosos en ML. Entender su funcionamiento interno es esencial para comprender y depurar Random Forest, Gradient Boosting y XGBoost.

---
<footer style="text-align:center; font-size:12px; color:gray;">
© 2026 UNAM Facultad de Ciencias — Inteligencia Artificial
</footer>